# Aureas · results viewer

Opens the folder of an Aureas run and shows its clusters **from the CSV files only**: it needs neither the dashboard, nor `run.yaml`, nor `arrays.npz`. It also works for runs whose export was interrupted, as long as `Classification_<date>.csv` exists (with or without `_part2`, `_part3`…), and for runs with millions of segments.

1. Set the run folder (next cell) and run everything (*Run All*).
2. **Summary** of the run and table of clusters.
3. **Gallery** with the representative element of every cluster.
4. **One cluster**: representative (spectrogram + audio) and its members.
5. **One recording**: spectrogram with the segments coloured by class.
6. **Activity** by hour and export of the summary.

To draw spectrograms and play audio, the notebook must be able to read the recordings. If they are now somewhere else than when Aureas was run (another computer or disk), set `AUDIO_FOLDER` to the folder where they are now: recordings are found by file name, also inside sub-folders.

In [ ]:
import os
from pathlib import Path

# ----------------------------------------------------------------- settings
# Folder of the run (the one containing Classification_<date>.csv)
RUN = Path(os.environ.get("AUREAS_RUN", "/path/to/results/<run_folder>"))

# >>> Folder where the recordings are NOW (searched by file name, also in sub-folders).
#     Leave None if they are still where they were when Aureas was run.
AUDIO_FOLDER = os.environ.get("AUREAS_AUDIO") or None
# e.g. AUDIO_FOLDER = "/Volumes/MyDisk/recordings"

# (advanced) replace path prefixes: {"old prefix": "new prefix"}
REPLACE_PATHS = {}

# Folder of aureas-py (only needed if the package is not installed in this environment)
AUREAS_PY = Path(os.environ.get("AUREAS_PY", ".."))

CHANNEL = 1        # audio channel (1 = first)

In [ ]:
import sys
if (AUREAS_PY / "aureas").exists() and str(AUREAS_PY.resolve()) not in sys.path:
    sys.path.insert(0, str(AUREAS_PY.resolve()))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import Audio, display, Markdown
from pandas.api.types import union_categoricals

from aureas.io import load_audio
from aureas.matlab_compat import spectrogram
from aureas.results import csv_parts

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
plt.rcParams["figure.dpi"] = 100
assert RUN.exists(), f"Folder not found: {RUN}"

## 1. Load the classification

Only the needed columns are read, with compact types (text as categories, 32-bit numbers), and all parts are joined, so millions of segments fit in a few hundred MB of RAM.

In [ ]:
COLS = {"SegmentID": "int64", "File": "category", "Path": "category", "Site": "category",
        "Hour": "float32", "Start": "float64", "End": "float64", "Length": "float32",
        "Fdom": "float32", "FminVoc": "float32", "FmaxVoc": "float32", "Class": "int32",
        "Membership": "float32", "MembershipAssigned": "float32",
        "IsRepresentative": "boolean", "Label": "category"}


def read_classification(run: Path) -> pd.DataFrame:
    import re
    bases = sorted(p for p in run.glob("Classification_*.csv") if not re.search(r"_parte?\d+$", p.stem))
    if not bases:
        raise FileNotFoundError(f"No Classification_*.csv in {run}")
    base = bases[-1]
    parts = csv_parts(base)
    header = pd.read_csv(base, nrows=0).columns
    use = [c for c in COLS if c in header]
    tables = []
    for i, p in enumerate(parts, 1):
        print(f"  reading {p.name} ({i}/{len(parts)})", flush=True)
        tables.append(pd.read_csv(p, usecols=use, dtype={c: COLS[c] for c in use}))
    # join the categories of all parts without turning them into text
    cats = {c: union_categoricals([t[c] for t in tables]) for c in use if COLS[c] == "category"}
    df = pd.concat([t.drop(columns=list(cats)) for t in tables], ignore_index=True)
    for c, v in cats.items():
        df[c] = v
    if "SegmentID" not in df:
        df["SegmentID"] = np.arange(len(df))
    if "MembershipAssigned" not in df and "Membership" in df:
        df["MembershipAssigned"] = df["Membership"]
    print(f"{base.name}: {len(parts)} part(s), {len(df):,} segments")
    return df.set_index("SegmentID", drop=False)


seg = read_classification(RUN)

# ---------------------------------------------------------- audio paths on this computer
EXT = {".wav", ".flac", ".mp3", ".ogg", ".aif", ".aiff"}


def _audio_index(folder):
    # file name -> list of paths inside `folder` (recursive)
    idx = {}
    for q in Path(folder).rglob("*"):
        if q.suffix.lower() in EXT and not q.name.startswith("._"):
            idx.setdefault(q.name, []).append(q)
    return idx


def _locate(old, idx):
    v = Path(old)
    cands = idx.get(v.name, [])
    if len(cands) == 1:
        return str(cands[0])
    if len(cands) > 1:                      # same name in several folders: compare folder names
        def common(q):
            n = 0
            for a, b in zip(reversed(q.parts[:-1]), reversed(v.parts[:-1])):
                if a != b:
                    break
                n += 1
            return n
        return str(max(cands, key=common))
    return old


if "Path" in seg:
    old_paths = list(seg["Path"].cat.categories)
    new_paths = []
    for p in old_paths:
        for old, new in REPLACE_PATHS.items():
            if p.startswith(old):
                p = new + p[len(old):]
                break
        new_paths.append(p)
    if AUDIO_FOLDER:
        assert Path(AUDIO_FOLDER).exists(), f"AUDIO_FOLDER not found: {AUDIO_FOLDER}"
        print("Looking for recordings in", AUDIO_FOLDER, "...")
        idx = _audio_index(AUDIO_FOLDER)
        new_paths = [_locate(p, idx) for p in new_paths]
    if len(set(new_paths)) == len(new_paths):
        seg["Path"] = seg["Path"].cat.rename_categories(new_paths)
    else:                                   # two old paths point to the same new file
        seg["Path"] = seg["Path"].map(dict(zip(old_paths, new_paths))).astype("category")
    found = sum(Path(p).exists() for p in set(new_paths))
    print(f"Recordings found: {found:,} of {len(set(new_paths)):,}")
    if found < len(set(new_paths)):
        missing = next(p for p in new_paths if not Path(p).exists())
        print(f"  e.g. not found: {missing}")
        print("  -> check AUDIO_FOLDER")

print(f"{seg.memory_usage(deep=True).sum() / 1e6:,.0f} MB in memory")
seg.head()

## 2. Run and cluster summary

In [ ]:
NAMES = {}
if (RUN / "labels.csv").exists():                      # names given in the dashboard
    lab = pd.read_csv(RUN / "labels.csv")
    NAMES = {int(c): n for c, n in zip(lab["Class"], lab["Label"]) if isinstance(n, str) and n}


def name_of(c):
    c = int(c)
    return NAMES.get(c, "NIC (class 1)" if c == 1 else f"Class {c}")


memb_rep = "Membership" if "Membership" in seg else "MembershipAssigned"
g = seg.groupby("Class", observed=True)
summary = pd.DataFrame({
    "segments": g.size(),
    "pct": 100 * g.size() / len(seg),
    "membership_median": g["MembershipAssigned"].median(),
    "pct_membership_>=0.9": g["MembershipAssigned"].apply(lambda s: 100 * (s >= 0.9).mean()),
    "fdom_median_Hz": g["Fdom"].median(),
    "fdom_p10": g["Fdom"].quantile(0.1),
    "fdom_p90": g["Fdom"].quantile(0.9),
    "fmin_median": g["FminVoc"].median(),
    "fmax_median": g["FmaxVoc"].median(),
    "duration_median_s": g["Length"].median() if "Length" in seg else (g["End"].median() - g["Start"].median()),
    "recordings": g["File"].nunique(),
    "representative": g[memb_rep].idxmax(),            # SegmentID with the highest membership
})
if "IsRepresentative" in seg and seg["IsRepresentative"].fillna(False).any():
    rep = seg[seg["IsRepresentative"].fillna(False)].groupby("Class")["SegmentID"].first()
    summary.loc[rep.index, "representative"] = rep
summary.insert(0, "name", [name_of(c) for c in summary.index])
summary["representative"] = summary["representative"].astype("int64")

n_nic = summary.loc[1, "segments"] if 1 in summary.index else 0
display(Markdown(f"**{len(seg):,} segments · {seg['File'].nunique():,} recordings · "
                 f"{(summary.index != 1).sum()} clusters + NIC** "
                 f"({n_nic:,} segments in the NIC, {100 * n_nic / len(seg):.1f} %)"))
summary.sort_values("segments", ascending=False).round(2)

In [ ]:
r = summary.drop(index=1, errors="ignore").sort_values("segments", ascending=False)
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].bar(range(len(r)), r["segments"], color="#4C72B0")
ax[0].set_yscale("log"); ax[0].set_xlabel("clusters (largest first)"); ax[0].set_ylabel("segments (log)")
ax[0].set_title("Cluster sizes")
ax[1].scatter(r["duration_median_s"], r["fdom_median_Hz"] / 1000,
              s=10 + 290 * np.sqrt(r["segments"] / r["segments"].max()),
              alpha=0.55, color="#4C72B0", edgecolor="k", lw=0.3)
for c, row in r.head(15).iterrows():
    ax[1].annotate(str(c), (row["duration_median_s"], row["fdom_median_Hz"] / 1000), fontsize=7)
ax[1].set_xscale("log"); ax[1].set_xlabel("median duration (s, log)"); ax[1].set_ylabel("median Fdom (kHz)")
ax[1].set_title("Clusters in frequency and duration (size = number of segments)")
plt.tight_layout(); plt.show()

## Helper functions: spectrograms and audio

In [ ]:
_PALETTE = [matplotlib.colors.to_hex(c) for n in ("tab20", "tab20b", "tab20c") for c in plt.get_cmap(n).colors]
_PALETTE = _PALETTE[0::2] + _PALETTE[1::2]


def class_colour(c):
    return "#8c8c8c" if int(c) == 1 else _PALETTE[(int(c) - 2) % len(_PALETTE)]


def _spec(path, t0, t1, nwin=1024, overlap=0.5):
    x, fs = load_audio(path, CHANNEL, max(0.0, t0), t1)
    if len(x) < nwin:
        x = np.pad(x, (0, nwin - len(x)))
    S, f, t, P = spectrogram(x, fs, nwin, int(nwin * overlap), nwin)
    return f, t + max(0.0, t0), 10 * np.log10(P + 1e-20), fs


def _draw(ax, f, t, db, fmin, fmax, cmap="Greys"):
    rows = (f >= fmin) & (f <= fmax)
    img = db[rows]
    vmax = np.percentile(img, 99.7)
    vmin = max(vmax - 50, np.percentile(img, 60))
    dt = (t[1] - t[0]) / 2 if len(t) > 1 else 0
    ax.imshow(img, origin="lower", aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax,
              extent=[t[0] - dt, t[-1] + dt, fmin / 1000, fmax / 1000], interpolation="nearest")


def show_segment(sid, context=1.5, listen=True, audio_pad=0.5):
    # Spectrogram of one segment in context (red box) and its audio.
    r = seg.loc[sid]
    try:
        f, t, db, fs = _spec(r.Path, r.Start - context, r.End + context, nwin=512, overlap=0.75)
    except Exception as e:
        print(f"Cannot open the recording {r.Path} ({type(e).__name__}). Check AUDIO_FOLDER.")
        return
    fig, ax = plt.subplots(figsize=(12, 3.2))
    _draw(ax, f, t, db, max(0, r.FminVoc - 2000), min(fs / 2, r.FmaxVoc + 2000), cmap="Blues")
    ax.add_patch(Rectangle((r.Start, r.FminVoc / 1000), r.End - r.Start, (r.FmaxVoc - r.FminVoc) / 1000,
                           fill=False, ec="#d62728", lw=1.6))
    ax.set_xlabel("Time (s)"); ax.set_ylabel("kHz")
    ax.set_title(f"Segment {sid} · {r.File} · {r.Start:.2f}-{r.End:.2f} s · Fdom {r.Fdom:,.0f} Hz · "
                 f"{name_of(r.Class)} · membership {r.MembershipAssigned:.2f}", fontsize=9)
    plt.tight_layout(); plt.show()
    if listen:
        x, fs = load_audio(r.Path, CHANNEL, max(0.0, r.Start - audio_pad), r.End + audio_pad)
        display(Audio(x, rate=fs))


def _thumbnails(ids, columns=6, title=None, height=1.6):
    ids = list(ids)
    if not ids:
        print("(no segments)"); return
    nrows = int(np.ceil(len(ids) / columns))
    fig, axs = plt.subplots(nrows, columns, figsize=(2.6 * columns, height * nrows + (0.4 if title else 0)),
                            squeeze=False)
    for ax in axs.flat:
        ax.set_axis_off()
    for ax, sid in zip(axs.flat, ids):
        r = seg.loc[sid]
        try:
            f, t, db, fs = _spec(r.Path, r.Start - 0.1, r.End + 0.1, nwin=512, overlap=0.75)
            _draw(ax, f, t, db, max(0, r.FminVoc - 300), min(fs / 2, r.FmaxVoc + 300), cmap="Blues")
        except Exception:
            ax.text(0.5, 0.5, "recording not\naccessible", ha="center", va="center", fontsize=8)
        ax.set_title(f"{sid} · {r.MembershipAssigned:.2f}", fontsize=7)
    if title:
        fig.suptitle(title, fontsize=10)
    plt.tight_layout(); plt.show()

## 3. Gallery of clusters

The representative element of every cluster (the one with the highest membership), largest clusters first. Above each thumbnail: `class · number of segments`. Change `N_GALLERY` or `ORDER` ("segments", "fdom_median_Hz").

In [ ]:
N_GALLERY = 48
ORDER = "segments"


def gallery(n=N_GALLERY, order=ORDER, columns=8, include_nic=False):
    r = summary if include_nic else summary.drop(index=1, errors="ignore")
    r = r.sort_values(order, ascending=(order != "segments")).head(n)
    nrows = int(np.ceil(len(r) / columns))
    fig, axs = plt.subplots(nrows, columns, figsize=(2.1 * columns, 1.7 * nrows), squeeze=False)
    for ax in axs.flat:
        ax.set_axis_off()
    for ax, (c, row) in zip(axs.flat, r.iterrows()):
        s = seg.loc[int(row["representative"])]
        try:
            f, t, db, fs = _spec(s.Path, s.Start - 0.1, s.End + 0.1, nwin=512, overlap=0.75)
            _draw(ax, f, t, db, max(0, s.FminVoc - 300), min(fs / 2, s.FmaxVoc + 300), cmap="Blues")
        except Exception:
            ax.text(0.5, 0.5, "recording not\naccessible", ha="center", va="center", fontsize=8)
        ax.set_title(f"{name_of(c)} · {int(row['segments']):,}", fontsize=8, color=class_colour(c))
    plt.tight_layout(); plt.show()


gallery()

## 4. One cluster in detail

Change `CLASS`. Shows its numbers, the representative (spectrogram in context + audio) and a grid of members:
`MEMBER_ORDER = "typical"` (highest membership first), `"borderline"` (lowest first) or `"random"`.

In [ ]:
CLASS = int(summary.drop(index=1, errors="ignore")["segments"].idxmax())   # the largest cluster
N_MEMBERS = 24
MEMBER_ORDER = "typical"


def show_cluster(cls=CLASS, n=N_MEMBERS, order=MEMBER_ORDER, seed=0):
    row = summary.loc[cls]
    display(Markdown(f"### {name_of(cls)}"))
    display(row.drop("name").to_frame("value").T.round(2))
    m = seg[seg["Class"] == cls]
    top = m["File"].value_counts().head(5)
    print("Recordings with most segments:", ", ".join(f"{k} ({v})" for k, v in top.items()))
    print("\nRepresentative:")
    show_segment(int(row["representative"]))
    if order == "random":
        ids = m.sample(min(n, len(m)), random_state=seed)["SegmentID"]
    else:
        ids = m.sort_values("MembershipAssigned", ascending=(order == "borderline"))["SegmentID"].head(n)
    _thumbnails(ids, title=f"{name_of(cls)}: {len(ids)} members ({order}) · title = SegmentID · membership")


show_cluster()

In [ ]:
# To see and hear any segment of the grid, use its SegmentID:
# show_segment(12345)

## 5. One recording with its segments coloured by class

Change `RECORDING` (file name) and the time window. Every segment is drawn in the colour of its class with its number; the NIC in dashed grey (hidden with `SHOW_NIC = False`).

In [ ]:
_ok = seg.drop_duplicates("File").set_index("File")["Path"].map(lambda p: Path(p).exists())
_vc = seg["File"].value_counts()
RECORDING = next((f for f in _vc.index if _ok.get(f, False)), _vc.index[0])   # accessible recording with most segments
WINDOW = (0, 30)                                 # seconds
FREQUENCY = (0, None)                            # Hz; None = up to fs/2
SHOW_NIC = False
CLASSES = None                                   # e.g. [5, 12, 43] to show only those


def show_recording(recording=RECORDING, window=WINDOW, frequency=FREQUENCY, show_nic=SHOW_NIC,
                   classes=CLASSES, listen=True):
    rs = seg[seg["File"] == recording]
    if rs.empty:
        print("No segments for", recording); return
    path = rs["Path"].iloc[0]
    a, b = window
    try:
        f, t, db, fs = _spec(path, a, b)
    except Exception as e:
        print(f"Cannot open the recording {path} ({type(e).__name__}). Check AUDIO_FOLDER.")
        return rs
    fmin, fmax = frequency[0], frequency[1] or fs / 2
    vis = rs[(rs["End"] >= a) & (rs["Start"] <= b)]
    if not show_nic:
        vis = vis[vis["Class"] != 1]
    if classes is not None:
        vis = vis[vis["Class"].isin(classes)]
    fig, ax = plt.subplots(figsize=(float(np.clip(6 + (b - a) * 0.12, 12, 22)), 4.6))
    _draw(ax, f, t, db, fmin, fmax)
    for r in vis.itertuples():
        col = class_colour(r.Class)
        nic = r.Class == 1
        ax.add_patch(Rectangle((r.Start, r.FminVoc / 1000), r.End - r.Start, (r.FmaxVoc - r.FminVoc) / 1000,
                               fc=(*matplotlib.colors.to_rgb(col), 0.12), ec=col,
                               lw=1.0 if nic else 1.6, ls="--" if nic else "-"))
        ax.text(r.Start, min(r.FmaxVoc, fmax) / 1000, str(r.Class), color=col, fontsize=7.5,
                fontweight="bold", va="bottom", clip_on=True)
    ax.set_xlim(a, b); ax.set_ylim(fmin / 1000, fmax / 1000)
    ax.set_xlabel("Time (s)"); ax.set_ylabel("kHz"); ax.set_title(f"{recording} · {len(vis)} segments", fontsize=10)
    plt.tight_layout(); plt.show()
    cnt = vis["Class"].value_counts()
    display(Markdown(" · ".join(f"<span style='color:{class_colour(c)}'>■</span> {name_of(c)}: {n}"
                                for c, n in cnt.items())))
    if listen:
        x, fs = load_audio(path, CHANNEL, a, b)
        display(Audio(x, rate=fs))
    return vis


window_segments = show_recording()

In [ ]:
# segments in the window (to see one: show_segment(SegmentID))
window_segments[["SegmentID", "Start", "End", "Fdom", "FminVoc", "FmaxVoc", "Class", "MembershipAssigned"]].head(30)

## 6. Activity by hour of the day

In [ ]:
ACTIVITY_CLUSTERS = list(summary.drop(index=1, errors="ignore").sort_values("segments", ascending=False).index[:8])

if "Hour" not in seg or seg["Hour"].fillna(0).eq(0).all():
    print("Recording times could not be read from the file names: no activity pattern.")
else:
    d = seg[seg["Class"].isin(ACTIVITY_CLUSTERS)]
    tab = d.groupby(["Hour", "Class"], observed=True).size().unstack(fill_value=0).reindex(range(24), fill_value=0)
    nrec = seg.groupby("Hour", observed=True)["File"].nunique().reindex(range(24)).replace(0, np.nan)
    tab = tab.div(nrec, axis=0)                                 # detections per recording
    fig, ax = plt.subplots(figsize=(14, 4))
    im = ax.imshow(tab.T.fillna(0).values, aspect="auto", cmap="viridis")
    ax.set_yticks(range(len(tab.columns)), [name_of(c) for c in tab.columns], fontsize=8)
    ax.set_xticks(range(24)); ax.set_xlabel("Hour of day")
    plt.colorbar(im, ax=ax, label="detections per recording")
    ax.set_title("Activity by hour (largest clusters)")
    plt.tight_layout(); plt.show()

## 7. Export the cluster summary

In [ ]:
out = RUN / "Cluster_summary_notebook.csv"
summary.to_csv(out, index_label="Class", encoding="utf-8-sig")
print("Saved:", out)